# Prompt Templates & Output Parsers

This notebook covers:
- `ChatPromptTemplate`: simple, multi-message, and composed prompts
- `MessagesPlaceholder`: injecting dynamic conversation history
- `FewShotChatMessagePromptTemplate`: few-shot prompting
- **Output Parsers**: `StrOutputParser`, `JsonOutputParser`, `PydanticOutputParser`
- **`with_structured_output()`**: the modern approach to structured data extraction

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. ChatPromptTemplate

### Simple template — `from_template`
Variables in `{curly_braces}` become the chain's input keys.

In [2]:
# Single-message template
simple = ChatPromptTemplate.from_template("Translate '{text}' to {language}")
messages = simple.format_messages(text="Hello, world!", language="French")
print("Simple template:")
for m in messages:
    print(f"  {type(m).__name__}: {m.content}")

Simple template:
  HumanMessage: Translate 'Hello, world!' to French


### Multi-message template — `from_messages`
Use tuples `(role, content)` to control which messages are `system`, `human`, or `ai`.

In [3]:
multi = ChatPromptTemplate.from_messages([
    ("system", "You are a translator. Be concise."),
    ("human", "Translate '{text}' to {language}"),
])

messages = multi.format_messages(text="Good morning", language="Japanese")
print("Multi-message template:")
for m in messages:
    print(f"  {type(m).__name__}: {m.content}")

Multi-message template:
  SystemMessage: You are a translator. Be concise.
  HumanMessage: Translate 'Good morning' to Japanese


### Prompt Composition — combining partial prompts

You can split prompts into reusable pieces and join them with `+`.

In [4]:
persona = ChatPromptTemplate.from_messages([("system", "You are a {role}. Your tone is {tone}.")] )
task    = ChatPromptTemplate.from_messages([("human", "{task}")])

# Combine
full_prompt = persona + task
chain = full_prompt | model | StrOutputParser()

resp = chain.invoke({"role": "pirate captain", "tone": "adventurous", "task": "Tell me about your ship"})
print(f"Pirate: {resp[:150]}...")

resp2 = chain.invoke({"role": "scientist", "tone": "precise", "task": "Explain photosynthesis"})
print(f"\nScientist: {resp2[:150]}...")

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 2. MessagesPlaceholder — Injecting Conversation History

`MessagesPlaceholder` reserves a slot in the template for a **list of messages** (typically your chat history). This is the key pattern for stateful chatbots.

In [ ]:
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),  # <-- injected at runtime
    ("human", "{question}"),
])

# Simulate history from a previous turn
history = [
    HumanMessage(content="My name is Paulo"),
    AIMessage(content="Nice to meet you, Paulo!"),
]

chain = prompt_with_history | model | StrOutputParser()
response = chain.invoke({"history": history, "question": "What's my name?"})
print(f"Response: {response}")

## 3. Few-Shot Prompting

Provide worked examples to guide the model's output format. `FewShotChatMessagePromptTemplate` wraps a list of examples and an example template.

In [ ]:
examples = [
    {"word": "happy", "opposite": "sad"},
    {"word": "fast",  "opposite": "slow"},
    {"word": "big",   "opposite": "small"},
]

# Template for each example pair
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What's the opposite of '{word}'?"),
    ("ai",    "The opposite of '{word}' is '{opposite}'."),
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You give the opposite of words. Follow the examples."),
    few_shot,
    ("human", "What's the opposite of '{word}'?"),
])

chain = final_prompt | model | StrOutputParser()
print(chain.invoke({"word": "bright"}))

## 4. Output Parsers

Parsers transform the LLM's raw `AIMessage` into a usable Python type.

| Parser | Output type | Use when |
|---|---|---|
| `StrOutputParser` | `str` | Simple text |
| `JsonOutputParser` | `dict` | Need a dict but no schema |
| `PydanticOutputParser` | Pydantic model | Schema needed + format instructions |
| `with_structured_output()` | Pydantic model | Modern, schema-only (recommended) |

In [ ]:
# StrOutputParser — returns a plain string
chain = ChatPromptTemplate.from_template("One-word answer: What color is the sky?") | model | StrOutputParser()
result = chain.invoke({})
print(f"StrOutputParser result: '{result}' (type: {type(result).__name__})")

In [ ]:
# JsonOutputParser — returns a dict
json_prompt = ChatPromptTemplate.from_template(
    "Return a JSON object with keys 'city' and 'country' for: {place}\n"
    "Return ONLY valid JSON, no explanation."
)
chain = json_prompt | model | JsonOutputParser()
result = chain.invoke({"place": "The Eiffel Tower"})
print(f"JsonOutputParser result: {result}")
print(f"City: {result['city']}, Country: {result['country']}")

In [ ]:
# PydanticOutputParser — injects format instructions into the prompt automatically
class Recipe(BaseModel):
    name: str = Field(description="Name of the recipe")
    ingredients: List[str] = Field(description="List of ingredients")
    prep_time_minutes: int = Field(description="Preparation time in minutes")
    difficulty: str = Field(description="easy, medium, or hard")

parser = PydanticOutputParser(pydantic_object=Recipe)

# .partial() pre-fills the {format_instructions} variable
prompt = ChatPromptTemplate.from_template(
    "Create a simple recipe for: {dish}\n\n{format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser
result = chain.invoke({"dish": "scrambled eggs"})

print(f"Name: {result.name}")
print(f"Ingredients: {result.ingredients}")
print(f"Prep time: {result.prep_time_minutes} mins  (type: {type(result.prep_time_minutes).__name__})")
print(f"Difficulty: {result.difficulty}")

## 5. `with_structured_output()` — Modern Structured Extraction

The recommended approach. Instead of injecting format instructions into the prompt, you bind a Pydantic schema directly to the model. The model uses **tool calling** under the hood — no format instructions in the prompt.

> Use this over `PydanticOutputParser` for production work.

In [ ]:
class TaskExtraction(BaseModel):
    """Extracted task information."""
    task: str = Field(description="The main task to do")
    priority: str = Field(description="high, medium, or low")
    deadline: Optional[str] = Field(description="Deadline if mentioned")
    assignee: Optional[str] = Field(description="Person assigned if mentioned")

# Bind schema to model — no format_instructions needed
structured_model = model.with_structured_output(TaskExtraction)

prompt = ChatPromptTemplate.from_template("Extract task information from: {text}")
chain = prompt | structured_model

texts = [
    "John needs to finish the report by Friday - it's urgent",
    "We should update the docs sometime next week",
    "Critical: Fix the login bug ASAP",
]

for text in texts:
    result = chain.invoke({"text": text})
    print(f"Input: {text}")
    print(f"  Task: {result.task}  |  Priority: {result.priority}  |  Deadline: {result.deadline}")
    print()

### Nested / complex schemas

In [ ]:
class Address(BaseModel):
    street: str
    city: str
    country: str

class Company(BaseModel):
    name: str
    industry: str
    employee_count: int
    headquarters: Address
    products: List[str]

chain = ChatPromptTemplate.from_template("Extract company information from: {text}") | model.with_structured_output(Company)

result = chain.invoke({"text": "Apple Inc. is a tech company with 160,000 employees based in Cupertino, California, USA. They make iPhones, MacBooks, and iPads."})

print(f"Company: {result.name}")
print(f"Industry: {result.industry}")
print(f"Employees: {result.employee_count:,}")
print(f"HQ: {result.headquarters.city}, {result.headquarters.country}")
print(f"Products: {result.products}")